# 02 · Sectoral Distribution — ΔSD


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

PROJECT_DIR = r'C:\Users\name\Documents\Bachelor Thesis\Empirical Analysis'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')

# Same-window counterfactual (see 00_data_ingestion.ipynb Steps 13-15):
# Original-today  = pre-reform universe,  pre-reform FFW,  2025-26 FF mkt cap
# Pro-forma-today = post-reform inclusion, post-reform FFW, 2025-26 FF mkt cap
orig = pd.read_parquet(os.path.join(DATA_DIR, 'constituents_original_today.parquet'))
pf   = pd.read_parquet(os.path.join(DATA_DIR, 'constituents_proforma.parquet'))

# FF_MktCap is NaN for quarantined rows, so a single `.notna()` filter
# yields the investable set in both files.
orig = orig[orig['FF_MktCap'].notna()].copy()
pf   = pf[pf['FF_MktCap'].notna()].copy()
orig['w'] = orig['FF_MktCap'] / orig['FF_MktCap'].sum()
pf['w']   = pf['FF_MktCap']  / pf['FF_MktCap'].sum()

print(f'Original-today : {len(orig):,}    Pro-forma-today : {len(pf):,}')


In [ ]:
# ── Sector mapping: TRBC Industry Group, with 2021 fallback for dropped stocks ──
# ~1,200 of the pre-reform stocks have left TOPIX and lost live Refinitiv
# coverage, so their TRBC_Industry at SDate=END_R is NaN. meta.parquet (Step 2
# of ingestion) fetched the same field at SDate=START_P=2021 when those stocks
# were still active — for stocks that have since left the index, the 2021
# classification is the correct one to use anyway.
SECTOR_COL = 'TRBC_Industry'

meta_hist_fp = os.path.join(DATA_DIR, 'meta.parquet')
if os.path.exists(meta_hist_fp):
    meta_hist = pd.read_parquet(meta_hist_fp)
    if 'RIC' in meta_hist.columns and SECTOR_COL in meta_hist.columns:
        hist_map = (meta_hist[['RIC', SECTOR_COL]]
                    .dropna(subset=[SECTOR_COL])
                    .drop_duplicates('RIC')
                    .set_index('RIC')[SECTOR_COL])
        mask_na = orig[SECTOR_COL].isna()
        n_before = int(mask_na.sum())
        orig.loc[mask_na, SECTOR_COL] = orig.loc[mask_na, 'RIC'].map(hist_map)
        n_after = int(orig[SECTOR_COL].isna().sum())
        print(f'Historical TRBC fill (orig): rescued {n_before - n_after} / {n_before} NaN stocks.')
    else:
        print('meta.parquet found but missing RIC or TRBC_Industry column — skipping fill.')
else:
    print('meta.parquet not found — no historical fill applied.')

orig[SECTOR_COL] = orig[SECTOR_COL].fillna('Unclassified')
pf[SECTOR_COL]   = pf[SECTOR_COL].fillna('Unclassified')
print(f'Sector coverage orig: {(orig[SECTOR_COL] != "Unclassified").sum()}/{len(orig)}')
print(f'Sector coverage pf  : {(pf[SECTOR_COL] != "Unclassified").sum()}/{len(pf)}')


In [ ]:
# ── Aggregate sector weights ──────────────────────────────────────────────────
w_orig = orig.groupby(SECTOR_COL)['w'].sum().rename('W_orig')
w_pf   = pf.groupby(SECTOR_COL)['w'].sum().rename('W_pf')

df_sec = pd.concat([w_orig, w_pf], axis=1).fillna(0)
df_sec['Delta']    = df_sec['W_pf'] - df_sec['W_orig']
df_sec['AbsDelta'] = df_sec['Delta'].abs()
df_sec = df_sec.sort_values('W_orig', ascending=False)

delta_sd = 0.5 * df_sec['AbsDelta'].sum()

print('Sector weights (original vs pro-forma):')
print(df_sec.applymap(lambda x: f'{x:.4%}' if isinstance(x, float) else x).to_string())
print(f'\nΔSD = {delta_sd:.4f}  ({delta_sd:.2%})')

In [ ]:
# ── Visualisation ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sectors = df_sec.index.tolist()
x, w = np.arange(len(sectors)), 0.35

# A: grouped horizontal bars
ax = axes[0]
ax.barh(x + w/2, df_sec['W_orig']*100, w, label='Original-today',  color='steelblue',  alpha=0.85)
ax.barh(x - w/2, df_sec['W_pf']*100,   w, label='Pro-forma-today', color='darkorange', alpha=0.85)
ax.set_yticks(x)
ax.set_yticklabels(sectors, fontsize=8)
ax.set_xlabel('Index weight (%)')
ax.set_title('A  Sector Weights: Original-today vs Pro-forma-today')
ax.legend()
ax.invert_yaxis()

# B: delta bars
ax = axes[1]
colors = ['darkorange' if d > 0 else 'steelblue' for d in df_sec['Delta']]
ax.barh(sectors, df_sec['Delta']*100, color=colors, alpha=0.85, edgecolor='black', lw=0.4)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Weight change (pp)')
ax.set_title(f'B  Δ Weight (Pro-forma-today − Original-today)   ΔSD = {delta_sd:.2%}')
ax.invert_yaxis()

plt.suptitle('TOPIX Reform — Sectoral Distribution (same-window counterfactual, 2025-26)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('sectoral_distribution.png', bbox_inches='tight')
plt.show()
print('Saved: sectoral_distribution.png')


In [ ]:
print(f'Key results:')
print(f'  ΔSD = {delta_sd:.4f}  ({delta_sd:.2%})')
print(f'  Sectors gaining weight : {(df_sec["Delta"] > 0).sum()}')
print(f'  Sectors losing weight  : {(df_sec["Delta"] < 0).sum()}')
print(f'  Largest gainer : {df_sec["Delta"].idxmax()}  ({df_sec["Delta"].max():+.3%})')
print(f'  Largest loser  : {df_sec["Delta"].idxmin()}  ({df_sec["Delta"].min():+.3%})')

df_sec.to_csv('sectoral_distribution_results.csv')
print('Saved: sectoral_distribution_results.csv')